In [1]:
import os
from io import StringIO

import pandas as pd
import psycopg2
from dotenv import load_dotenv
from psycopg2 import sql

load_dotenv()

DEFAULT_SCHEMA = "athl_v2"

def _build_local_conn_string() -> str:
    host = os.getenv("POSTGRES_HOST", "localhost")
    port = os.getenv("POSTGRES_PORT", "5432")
    db = os.getenv("POSTGRES_DB", "analytics_db")
    user = os.getenv("POSTGRES_USER", "admin")
    password = os.getenv("POSTGRES_PASSWORD", "admin")
    return f"postgresql://{user}:{password}@{host}:{port}/{db}?sslmode=disable"

conn_string = os.getenv("LOCAL_DATABASE_URL") or _build_local_conn_string()
print("Using local Postgres connection. Default schema:", DEFAULT_SCHEMA)


Using local Postgres connection. Default schema: athl_v2


In [2]:
import os
from io import StringIO

import pandas as pd
import psycopg2
from dotenv import load_dotenv
from psycopg2 import sql

load_dotenv()
host = os.getenv("POSTGRES_HOST", "localhost")
port = os.getenv("POSTGRES_PORT", "5432")
db = os.getenv("POSTGRES_DB", "analytics_db")
user = os.getenv("POSTGRES_USER", "admin")
password = os.getenv("POSTGRES_PASSWORD", "admin")

print 
(host,
port,
db,
user,
password)

('localhost', '5432', 'analytics_db', 'admin', 'admin')

In [3]:
def truncate_data(conn_string: str, table_name: str, schema: str = DEFAULT_SCHEMA, cascade: bool = False):
    """Truncate a table in local Postgres athl_v2 (or provided schema)."""
    suffix = " CASCADE" if cascade else ""
    truncate_sql = sql.SQL("TRUNCATE TABLE {}.{} RESTART IDENTITY" + suffix).format(
        sql.Identifier(schema),
        sql.Identifier(table_name),
    )

    with psycopg2.connect(conn_string) as conn:
        with conn.cursor() as cur:
            cur.execute(truncate_sql)
        conn.commit()
    print(f"Truncated {schema}.{table_name}")


def copy_data(conn_string: str, df: pd.DataFrame, table_name: str, schema: str = DEFAULT_SCHEMA):
    """COPY a DataFrame into local Postgres athl_v2 (or provided schema)."""
    buf = StringIO()
    df.to_csv(buf, index=False, header=True)
    buf.seek(0)

    columns = [sql.Identifier(col) for col in df.columns]
    copy_sql = sql.SQL("COPY {}.{} ({}) FROM STDIN WITH (FORMAT csv, HEADER true)").format(
        sql.Identifier(schema),
        sql.Identifier(table_name),
        sql.SQL(", ").join(columns),
    )

    with psycopg2.connect(conn_string) as conn:
        with conn.cursor() as cur:
            cur.copy_expert(copy_sql, buf)
        conn.commit()

    print(f"Loaded {len(df)} rows into {schema}.{table_name}")


In [ ]:
# Example usage:
# truncate_data(conn_string, "athlete_profile", cascade=True)
# copy_data(conn_string, athlete_profile_df, "athlete_profile")


In [4]:
# File names
# folder_path = os.getcwd()  # change if needed
folder_path = '/Users/alikhanzadi/Desktop/Learn/Projects/athl-data-platform/data/tables'

file_names = [os.path.splitext(f)[0] for f in os.listdir(folder_path) if f.endswith(".csv")]

print (file_names)

['athlete_profile', 'social_verification', 'issuer_daily_revenue', 'users', 'user_wallet', 'issuer_post_signup', 'issuers', 'issuer_preferences', 'user_token_wallet', 'identity_verification', 'transactions', 'tokens']


In [5]:
for file in file_names:
    truncate_data(conn_string, file, cascade=True)

Truncated athl_v2.athlete_profile
Truncated athl_v2.social_verification
Truncated athl_v2.issuer_daily_revenue
Truncated athl_v2.users
Truncated athl_v2.user_wallet
Truncated athl_v2.issuer_post_signup
Truncated athl_v2.issuers
Truncated athl_v2.issuer_preferences
Truncated athl_v2.user_token_wallet
Truncated athl_v2.identity_verification
Truncated athl_v2.transactions
Truncated athl_v2.tokens


In [6]:
# Enforce dependency-safe load order (parents before children)
preferred_order = [
    "users",
    "issuers",
    "identity_verification",
    "social_verification",
    "athlete_profile",
    "issuer_post_signup",
    "issuer_preferences",
    "tokens",
    "transactions",
    "user_token_wallet",
    "user_wallet",
    "issuer_daily_revenue",
]

# present = set(file_names)
# ordered_existing = [t for t in preferred_order if t in present]
# remaining = sorted([t for t in file_names if t not in set(ordered_existing)])

# file_names = ordered_existing + remaining
# print("Safe load order:", file_names)

In [7]:
for file in preferred_order:
    df = pd.read_csv(f"{folder_path}/{file}.csv")
    copy_data(conn_string, df, file)

Loaded 2000 rows into athl_v2.users
Loaded 200 rows into athl_v2.issuers
Loaded 200 rows into athl_v2.identity_verification
Loaded 200 rows into athl_v2.social_verification
Loaded 107 rows into athl_v2.athlete_profile
Loaded 200 rows into athl_v2.issuer_post_signup
Loaded 200 rows into athl_v2.issuer_preferences
Loaded 150 rows into athl_v2.tokens
Loaded 81844 rows into athl_v2.transactions
Loaded 66976 rows into athl_v2.user_token_wallet
Loaded 2000 rows into athl_v2.user_wallet
Loaded 35383 rows into athl_v2.issuer_daily_revenue
